This notebook builds the weekly Theater District, control zone, and HVFHV datasets and joins them with the Broadway grosses data.

In [33]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pandas as pd

spark = (SparkSession.builder
    .appName("Broadway_Weekly_Grosses")
    .config("spark.sql.repl.eagerEval.enabled", True) 
    .getOrCreate()
)

Read the weekly grosses file.

In [34]:
weekly_grosses = spark.read.csv(
    "../data/broadway_data/Weekly_grosses.csv",
    header=True,
    inferSchema=False
)

weekly_grosses.count()

2259

Check schema.

In [35]:
weekly_grosses.printSchema()

root
 |-- ID: string (nullable = true)
 |-- Week Ending: string (nullable = true)
 |-- Gross: string (nullable = true)
 |-- Attendance: string (nullable = true)
 |-- % Capacity: string (nullable = true)
 |-- _c5: string (nullable = true)



Change datatypes to correct ones and clean unwanted symbols.

In [36]:
weekly_clean = (
    weekly_grosses
    .withColumn(
        "Week Ending",
        F.to_date(F.col("Week Ending"), "MMM d, yyyy")
    )
    .withColumn(
        "Gross",
        F.regexp_replace(F.col("Gross"), "[$,]", "").cast("double")
    )
    .withColumn(
        "Attendance",
        F.regexp_replace(F.col("Attendance"), ",", "").cast("double")
    )
    .withColumn(
        "% Capacity",
        F.regexp_replace(F.col("% Capacity"), "%", "").cast("double") / 100
    )
)
weekly_clean.printSchema()

root
 |-- ID: string (nullable = true)
 |-- Week Ending: date (nullable = true)
 |-- Gross: double (nullable = true)
 |-- Attendance: double (nullable = true)
 |-- % Capacity: double (nullable = true)
 |-- _c5: string (nullable = true)



Check that filter did not turn any relevant data into NULL.

In [37]:
weekly_clean.filter(
    (F.year("Week Ending").isin([2023, 2024])) &
    F.col("Week Ending").isNull()
).count()

0

Filter to include 2023 and 2024 data.

In [38]:
weekly_grosses = weekly_clean.filter(
    F.year("Week Ending").isin([2023, 2024])
)

weekly_grosses.count()

105

Check for any gaps by checking if the difference between each instance is 7 days.

In [39]:
from pyspark.sql.window import Window

window = Window.orderBy("Week Ending")

weekly_check = weekly_grosses.withColumn(
    "previous_week",
    F.lag("Week Ending").over(window)
).withColumn(
    "days_since_previous",
    F.datediff("Week Ending", "previous_week")
)

weekly_check.filter(
    F.col("days_since_previous") != 7
).count()

26/08/31 19:05:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/31 19:05:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/31 19:05:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/31 19:05:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


0

Rename columns to join with tlc data.

In [40]:
weekly_grosses = (
    weekly_grosses
    .withColumnRenamed("Week Ending", "week_ending")
    .withColumnRenamed("Attendance", "attendance")
    .withColumnRenamed("Gross", "gross")
)

Load the curated Yellow Taxi data.

In [41]:
trips = spark.read.parquet(
    "../data/curated/yellow_taxi"
)
trips.printSchema()

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)



Add a week_ending column so trips can be grouped by week.

In [42]:
trips_with_week = trips.withColumn(
    "week_ending",
    F.to_date(
        F.date_add(
            F.to_date("tpep_pickup_datetime"),
            6 - ((F.dayofweek("tpep_pickup_datetime") + 5) % 7) # formula ensures that the week ends on that week's Sunday
        )
    )
)
trips_with_week  = trips_with_week.filter(
    (F.col("week_ending") >= "2023-01-01") &
    (F.col("week_ending") <= "2024-12-31")
)
trips_with_week.printSchema()

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- week_ending: date (nullable = true)



Check the week_ending column looks right.

In [43]:
trips_with_week.select(
    "tpep_pickup_datetime",
    "week_ending"
).orderBy(
    "tpep_pickup_datetime"
).limit(5)

tpep_pickup_datetime,week_ending
2023-01-01 00:03:04,2023-01-01
2023-01-01 00:03:04,2023-01-01
2023-01-01 00:03:25,2023-01-01
2023-01-01 00:04:15,2023-01-01
2023-01-01 00:04:26,2023-01-01


Aggregate trips to weekly totals per zone.

In [44]:
weekly_trips = trips_with_week.groupBy(
    "week_ending",
    "PULocationID"
).agg(
    F.count("*").alias("trip_count"),
    F.sum("fare_amount").alias("total_fare"),
    F.sum("tip_amount").alias("total_tip")
)
weekly_trips.printSchema()
weekly_trips.count()

root
 |-- week_ending: date (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- trip_count: long (nullable = false)
 |-- total_fare: double (nullable = true)
 |-- total_tip: double (nullable = true)



420

Load the theater zone IDs.

In [45]:
theaters_zones = pd.read_csv(
    "../data/broadway_data/theaters_zones.csv"
)

theater_zone_ids = (
    theaters_zones["LocationID"]
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)

theater_zone_ids

[230, 48, 100, 142]

Filter to theater zones and add up to one weekly total.

In [46]:
theater_weekly = (
    weekly_trips
    .filter(F.col("PULocationID").isin(theater_zone_ids))
    .filter(
        (F.col("week_ending") >= "2023-01-08") &
        (F.col("week_ending") <= "2024-12-29")
    )
    .groupBy("week_ending")
    .agg(
        F.sum("trip_count").alias("weekly_trips"),
        F.sum("total_fare").alias("weekly_fares"),
        F.sum("total_tip").alias("weekly_tips")
    )
    .orderBy("week_ending")
)

Check we got 104 weeks (52 for 2023 + 52 for 2024).

In [47]:
theater_weekly.count()

104

Look at the first few rows.

In [48]:
theater_weekly.show(5)

+-----------+------------+------------------+------------------+
|week_ending|weekly_trips|      weekly_fares|       weekly_tips|
+-----------+------------+------------------+------------------+
| 2023-01-08|       66721|1059553.7599999988|187122.48000000019|
| 2023-01-15|       78114|1177701.3599999954|228002.39000000193|
| 2023-01-22|       76255| 1188576.169999997|232831.13000000216|
| 2023-01-29|       75684| 1159828.029999998|225532.54000000114|
| 2023-02-05|       74676| 1141269.950000001|220014.48000000033|
+-----------+------------+------------------+------------------+
only showing top 5 rows


Join the weekly trips with the weekly grosses.

In [49]:
weekly_joined = theater_weekly.join(
    weekly_grosses,
    on="week_ending",
    how="inner"
)

Check the join kept all 104 weeks.

In [50]:
weekly_joined.count()

104

Look at the joined data.

In [51]:
weekly_joined.orderBy("week_ending").show(10, truncate=False)

26/08/31 19:05:32 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: ID, Week Ending, Gross, Attendance, % Capacity, 
 Schema: ID, Week Ending, Gross, Attendance, % Capacity, _c5
Expected: _c5 but found: 
CSV file: file:///mnt/c/Users/caiji/Downloads/ADS_Project_1/data/broadway_data/Weekly_grosses.csv


+-----------+------------+------------------+------------------+---+-----------+----------+----------+----+
|week_ending|weekly_trips|weekly_fares      |weekly_tips       |ID |gross      |attendance|% Capacity|_c5 |
+-----------+------------+------------------+------------------+---+-----------+----------+----------+----+
|2023-01-08 |66721       |1059553.7599999988|187122.48000000019|110|3.7394931E7|275834.0  |0.92      |NULL|
|2023-01-15 |78114       |1177701.3599999954|228002.39000000193|109|3.3859988E7|247667.0  |0.93      |NULL|
|2023-01-22 |76255       |1188576.169999997 |232831.13000000216|108|2.5835362E7|204847.0  |0.94      |NULL|
|2023-01-29 |75684       |1159828.029999998 |225532.54000000114|107|2.5705039E7|206578.0  |0.94      |NULL|
|2023-02-05 |74676       |1141269.950000001 |220014.48000000033|106|2.3518702E7|195520.0  |0.93      |NULL|
|2023-02-12 |76784       |1199817.5699999966|231624.84000000078|105|2.3064393E7|192323.0  |0.95      |NULL|
|2023-02-19 |79478       |12

Convert to pandas for plotting.

In [52]:
plot_data = weekly_joined.orderBy("week_ending").toPandas()

26/08/31 19:05:34 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: ID, Week Ending, Gross, Attendance, % Capacity, 
 Schema: ID, Week Ending, Gross, Attendance, % Capacity, _c5
Expected: _c5 but found: 
CSV file: file:///mnt/c/Users/caiji/Downloads/ADS_Project_1/data/broadway_data/Weekly_grosses.csv


Preview the data.

In [53]:
plot_data.head()

,week_ending,weekly_trips,weekly_fares,weekly_tips,ID,gross,attendance,% Capacity,_c5
0,2023-01-08,66721,1059553.76,187122.48,110,37394931.0,275834.0,0.92,None
1,2023-01-15,78114,1177701.36,228002.39,109,33859988.0,247667.0,0.93,None
2,2023-01-22,76255,1188576.17,232831.13,108,25835362.0,204847.0,0.94,None
3,2023-01-29,75684,1159828.03,225532.54,107,25705039.0,206578.0,0.94,None
4,2023-02-05,74676,1141269.95,220014.48,106,23518702.0,195520.0,0.93,None


Control zone: repeat the same steps for Murray Hill.

In [54]:
control_trips = spark.read.parquet("../data/curated/control_zone")

control_zone_name = "Murray Hill"

control_with_week = control_trips.withColumn(
    "week_ending",
    F.to_date(
        F.date_add(
            F.to_date("tpep_pickup_datetime"),
            6 - ((F.dayofweek("tpep_pickup_datetime") + 5) % 7)
        )
    )
).filter(
    (F.col("week_ending") >= "2023-01-08") &
    (F.col("week_ending") <= "2024-12-29")
)

control_weekly = (
    control_with_week
    .groupBy("week_ending")
    .agg(F.count("*").alias("weekly_trips"))
    .orderBy("week_ending")
)

control_joined = control_weekly.join(
    weekly_grosses, on="week_ending", how="inner"
)
control_plot_data = control_joined.orderBy("week_ending").toPandas()
control_plot_data.head()

26/08/31 19:05:36 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: ID, Week Ending, Gross, Attendance, % Capacity, 
 Schema: ID, Week Ending, Gross, Attendance, % Capacity, _c5
Expected: _c5 but found: 
CSV file: file:///mnt/c/Users/caiji/Downloads/ADS_Project_1/data/broadway_data/Weekly_grosses.csv


,week_ending,weekly_trips,ID,gross,attendance,% Capacity,_c5
0,2023-01-08,17190,110,37394931.0,275834.0,0.92,None
1,2023-01-15,20255,109,33859988.0,247667.0,0.93,None
2,2023-01-22,20232,108,25835362.0,204847.0,0.94,None
3,2023-01-29,20799,107,25705039.0,206578.0,0.94,None
4,2023-02-05,20275,106,23518702.0,195520.0,0.93,None


HVFHV: repeat the same steps.

In [55]:
fhvhv_trips = spark.read.parquet("../data/curated/fhvhv")
fhvhv_trips.printSchema()


root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- originating_base_num: string (nullable = true)
 |-- request_datetime: timestamp_ntz (nullable = true)
 |-- on_scene_datetime: timestamp_ntz (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_time: long (nullable = true)
 |-- base_passenger_fare: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- bcf: double (nullable = true)
 |-- sales_tax: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- driver_pay: double (nullable = true)
 |-- shared_request_flag: string (nullable = true)
 |-- shared_match_flag: string (nullable = true)
 |-- access_a_ride_f

Add week_ending, filter to theater zones, and join with grosses.

In [56]:
fhvhv_with_week = fhvhv_trips.withColumn(
    "week_ending",
    F.to_date(
        F.date_add(
            F.to_date("pickup_datetime"),
            6 - ((F.dayofweek("pickup_datetime") + 5) % 7)
        )
    )
).filter(
    (F.col("week_ending") >= "2023-01-08") &
    (F.col("week_ending") <= "2024-12-29")
)
fhvhv_weekly = (
    fhvhv_with_week
    .filter(F.col("PULocationID").isin(theater_zone_ids))
    .groupBy("week_ending")
    .agg(
        F.count("*").alias("weekly_trips"),
        F.sum("base_passenger_fare").alias("weekly_fares"),
        F.sum("tips").alias("weekly_tips")
    )
    .orderBy("week_ending")
)
fhvhv_joined = fhvhv_weekly.join(
    weekly_grosses, on="week_ending", how="inner"
)
fhvhv_plot_data = fhvhv_joined.orderBy("week_ending").toPandas()
fhvhv_plot_data.head()

26/08/31 19:05:41 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: ID, Week Ending, Gross, Attendance, % Capacity, 
 Schema: ID, Week Ending, Gross, Attendance, % Capacity, _c5
Expected: _c5 but found: 
CSV file: file:///mnt/c/Users/caiji/Downloads/ADS_Project_1/data/broadway_data/Weekly_grosses.csv


,week_ending,weekly_trips,weekly_fares,weekly_tips,ID,gross,attendance,% Capacity,_c5
0,2023-01-08,130241,3304340.74,223230.93,110,37394931.0,275834.0,0.92,None
1,2023-01-15,158041,3968268.90,272688.69,109,33859988.0,247667.0,0.93,None
2,2023-01-22,165953,4297682.57,307612.14,108,25835362.0,204847.0,0.94,None
3,2023-01-29,162818,4206784.91,271745.62,107,25705039.0,206578.0,0.94,None
4,2023-02-05,160869,4158695.25,256000.14,106,23518702.0,195520.0,0.93,None


Look at pickups by hour of day and day of week, to check for a post-show spike or matinee-day pattern. (Manitee days are days with afternoon shows)

In [57]:
theater_hourly = (
    trips_with_week
    .filter(F.col("PULocationID").isin(theater_zone_ids))
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .withColumn("pickup_dow", F.date_format("tpep_pickup_datetime", "E"))  # Mon, Tue, ...
    .groupBy("pickup_hour")
    .agg(F.count("*").alias("trip_count"))
    .withColumn("zone_type", F.lit("theater_district"))
    .orderBy("pickup_hour")
)

control_hourly = (
    control_with_week
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .groupBy("pickup_hour")
    .agg(F.count("*").alias("trip_count"))
    .withColumn("zone_type", F.lit("control_zone"))
    .orderBy("pickup_hour")
)

hourly_pattern = theater_hourly.unionByName(control_hourly)
hourly_pattern_pd = hourly_pattern.toPandas()
hourly_pattern_pd.head()

,pickup_hour,trip_count,zone_type
0,0,244624,theater_district
1,1,139598,theater_district
2,2,85940,theater_district
3,3,62653,theater_district
4,4,57662,theater_district


Same thing but by day of week, and flag the matinee days.

In [58]:
theater_dow = (
    trips_with_week
    .filter(F.col("PULocationID").isin(theater_zone_ids))
    .withColumn("pickup_dow", F.date_format("tpep_pickup_datetime", "E"))
    .withColumn("is_matinee_day", F.col("pickup_dow").isin(["Wed", "Sat", "Sun"]))
    .groupBy("pickup_dow", "is_matinee_day")
    .agg(F.count("*").alias("trip_count"))
    .withColumn("zone_type", F.lit("theater_district"))
)

control_dow = (
    control_with_week
    .withColumn("pickup_dow", F.date_format("tpep_pickup_datetime", "E"))
    .withColumn("is_matinee_day", F.col("pickup_dow").isin(["Wed", "Sat", "Sun"]))
    .groupBy("pickup_dow", "is_matinee_day")
    .agg(F.count("*").alias("trip_count"))
    .withColumn("zone_type", F.lit("control_zone"))
)

dow_pattern = theater_dow.unionByName(control_dow)
dow_pattern_pd = dow_pattern.toPandas()
dow_pattern_pd.head(20)

,pickup_dow,is_matinee_day,trip_count,zone_type
0,Mon,False,977274,theater_district
1,Sat,True,1237857,theater_district
2,Thu,False,1288508,theater_district
3,Tue,False,1183392,theater_district
4,Sun,True,1048109,theater_district
5,Fri,False,1197366,theater_district
6,Wed,True,1280786,theater_district
7,Mon,False,279456,control_zone
8,Sat,True,282782,control_zone
9,Thu,False,381176,control_zone


FHVHV control zone: same steps again to compare against the Theater District like Yellow Taxi.

In [59]:
fhvhv_control_trips = spark.read.parquet("../data/curated/fhvhv_control_zone")
fhvhv_control_with_week = fhvhv_control_trips.withColumn(   
    "week_ending",
    F.to_date(
        F.date_add(
            F.to_date("pickup_datetime"),
            6 - ((F.dayofweek("pickup_datetime") + 5) % 7)
        )
    )
).filter(
    (F.col("week_ending") >= "2023-01-08") &
    (F.col("week_ending") <= "2024-12-29")
)

fhvhv_control_weekly = (
    fhvhv_control_with_week
    .groupBy("week_ending")
    .agg(F.count("*").alias("weekly_trips"))
    .orderBy("week_ending")
)

fhvhv_control_joined = fhvhv_control_weekly.join(weekly_grosses, on="week_ending", how="inner")
fhvhv_control_plot_data = fhvhv_control_joined.orderBy("week_ending").toPandas()
fhvhv_control_plot_data.head()

26/08/31 19:05:51 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: ID, Week Ending, Gross, Attendance, % Capacity, 
 Schema: ID, Week Ending, Gross, Attendance, % Capacity, _c5
Expected: _c5 but found: 
CSV file: file:///mnt/c/Users/caiji/Downloads/ADS_Project_1/data/broadway_data/Weekly_grosses.csv


,week_ending,weekly_trips,ID,gross,attendance,% Capacity,_c5
0,2023-01-08,33143,110,37394931.0,275834.0,0.92,None
1,2023-01-15,41316,109,33859988.0,247667.0,0.93,None
2,2023-01-22,42756,108,25835362.0,204847.0,0.94,None
3,2023-01-29,44851,107,25705039.0,206578.0,0.94,None
4,2023-02-05,45463,106,23518702.0,195520.0,0.93,None


Some of the four theater zones may only contain one or two theaters, meaning most of their trip volume is generic activity, not Broadway activity. Pick out whichever zone has the most theaters and rebuild the weekly series using only that zone, so it can be compared against the full four-zone aggregate as it can be seen that only one zone has more than 2 theaters.

In [60]:
theaters_zones = pd.read_csv("../data/broadway_data/theaters_zones.csv")
theater_counts = theaters_zones["LocationID"].value_counts().sort_values(ascending=False)

core_zone_id = int(theater_counts.index[0])
core_zone_theater_count = int(theater_counts.iloc[0])
print(f"Most theater-dense zone: {core_zone_id} ({core_zone_theater_count} theaters)")
print(f"Compare against the full theater zone list: {theater_zone_ids} "
      f"({theaters_zones['LocationID'].value_counts().to_dict()})")


Most theater-dense zone: 230 (37 theaters)
Compare against the full theater zone list: [230, 48, 100, 142] ({230: 37, 100: 2, 48: 1, 142: 1})


Build the weekly series for just the core zone and join with grosses.

In [61]:
core_weekly = (
    weekly_trips
    .filter(F.col("PULocationID") == core_zone_id)
    .filter((F.col("week_ending") >= "2023-01-08") & (F.col("week_ending") <= "2024-12-29"))
    .groupBy("week_ending")
    .agg(F.sum("trip_count").alias("weekly_trips"))
    .orderBy("week_ending")
)

core_joined = core_weekly.join(weekly_grosses, on="week_ending", how="inner")
core_plot_data = core_joined.orderBy("week_ending").toPandas()
print(f"Core zone weeks: {len(core_plot_data)} (expected 104)")
core_plot_data.head()

26/08/31 19:05:54 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: ID, Week Ending, Gross, Attendance, % Capacity, 
 Schema: ID, Week Ending, Gross, Attendance, % Capacity, _c5
Expected: _c5 but found: 
CSV file: file:///mnt/c/Users/caiji/Downloads/ADS_Project_1/data/broadway_data/Weekly_grosses.csv


Core zone weeks: 104 (expected 104)


,week_ending,weekly_trips,ID,gross,attendance,% Capacity,_c5
0,2023-01-08,19373,110,37394931.0,275834.0,0.92,None
1,2023-01-15,24455,109,33859988.0,247667.0,0.93,None
2,2023-01-22,23126,108,25835362.0,204847.0,0.94,None
3,2023-01-29,21950,107,25705039.0,206578.0,0.94,None
4,2023-02-05,22537,106,23518702.0,195520.0,0.93,None


Save everything to CSV.

In [62]:
import os
analysis_ready_dir = "../data/analysis_ready"
os.makedirs(analysis_ready_dir, exist_ok=True)
plot_data.to_csv(f"{analysis_ready_dir}/theater_district_weekly.csv", index=False)
core_plot_data.to_csv(f"{analysis_ready_dir}/core_zone_weekly.csv", index=False)
control_plot_data.to_csv(f"{analysis_ready_dir}/control_zone_weekly.csv", index=False)
fhvhv_plot_data.to_csv(f"{analysis_ready_dir}/fhvhv_theater_district_weekly.csv", index=False)
fhvhv_control_plot_data.to_csv(f"{analysis_ready_dir}/fhvhv_control_zone_weekly.csv", index=False)
print("Saved:")
print(f"  {analysis_ready_dir}/theater_district_weekly.csv       ({len(plot_data)} rows)")
print(f"  {analysis_ready_dir}/control_zone_weekly.csv           ({len(control_plot_data)} rows)")
print(f"  {analysis_ready_dir}/fhvhv_theater_district_weekly.csv ({len(fhvhv_plot_data)} rows)")
print(f"  {analysis_ready_dir}/fhvhv_control_zone_weekly.csv     ({len(fhvhv_control_plot_data)} rows)")

hourly_pattern_pd.to_csv(f"{analysis_ready_dir}/hourly_pattern.csv", index=False)
dow_pattern_pd.to_csv(f"{analysis_ready_dir}/dow_pattern.csv", index=False)
print(f"  {analysis_ready_dir}/hourly_pattern.csv")
print(f"  {analysis_ready_dir}/dow_pattern.csv")

Saved:
  ../data/analysis_ready/theater_district_weekly.csv       (104 rows)
  ../data/analysis_ready/control_zone_weekly.csv           (104 rows)
  ../data/analysis_ready/fhvhv_theater_district_weekly.csv (104 rows)
  ../data/analysis_ready/fhvhv_control_zone_weekly.csv     (104 rows)
  ../data/analysis_ready/hourly_pattern.csv
  ../data/analysis_ready/dow_pattern.csv


Check whether the Christmas-week dip shows up in the control zone too.

In [63]:
# check last few weeks for both zones
print(plot_data[["week_ending", "weekly_trips", "attendance"]].tail(4))
print()
print(control_plot_data[["week_ending", "weekly_trips"]].tail(4))

    week_ending  weekly_trips  attendance
100  2024-12-08         94589    309802.0
101  2024-12-15        100737    314951.0
102  2024-12-22         94497    302828.0
103  2024-12-29         63124    306180.0

    week_ending  weekly_trips
100  2024-12-08         24955
101  2024-12-15         26200
102  2024-12-22         23660
103  2024-12-29         13125


Check the last week to see if it's a real dip and not a data gap.

In [64]:
# daily breakdown of the last week
last_week_daily = (
    trips_with_week
    .filter(F.col("PULocationID").isin(theater_zone_ids))
    .filter(F.col("week_ending") == "2024-12-29")
    .groupBy(F.to_date("tpep_pickup_datetime").alias("pickup_date"))
    .agg(F.count("*").alias("daily_trips"))
    .orderBy("pickup_date")
)
last_week_daily.show()

+-----------+-----------+
|pickup_date|daily_trips|
+-----------+-----------+
| 2024-12-23|      10494|
| 2024-12-24|       8943|
| 2024-12-25|       6116|
| 2024-12-26|       8952|
| 2024-12-27|       9206|
| 2024-12-28|      10625|
| 2024-12-29|       8788|
+-----------+-----------+

